- Remove non alphanumeric characters for simple training

In [1]:
from transformer import Transformer # this is the transformer.py file
import torch
import numpy as np

In [2]:
english_file = 'en-mr/train.en' # replace this path with appropriate one
marathi_file = 'en-mr/train.mr' # replace this path with appropriate one

# Generated this by filtering Appendix code

START_TOKEN = '<START>'
PADDING_TOKEN = '<PADDING>'
END_TOKEN = '<END>'

marathi_vocabulary = [START_TOKEN, ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', 
                      '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', '<', '=', '>', '?', 'ˌ',

                      # Swar (Vowels)
                      'अ', 'आ', 'इ', 'ई', 'उ', 'ऊ', 'ऋ', 'ॠ', 'ऌ', 'ॡ', 'ए', 'ऐ', 'ओ', 'औ',

                      # Vyanjan (Consonants)
                      'क', 'ख', 'ग', 'घ', 'ङ',
                      'च', 'छ', 'ज', 'झ', 'ञ',
                      'ट', 'ठ', 'ड', 'ढ', 'ण',
                      'त', 'थ', 'द', 'ध', 'न',
                      'प', 'फ', 'ब', 'भ', 'म',
                      'य', 'र', 'ऱ', 'ल', 'ळ', 'व',
                      'श', 'ष', 'स', 'ह',

                      # Nukta letters (Marathi specific sounds)
                      'क़', 'ख़', 'ग़', 'ज़', 'ड़', 'ढ़', 'फ़', 'य़',

                      # Matras (Vowel signs)
                      'ा', 'ि', 'ी', 'ु', 'ू', 'ृ', 'ॄ', 'ॅ', 'े', 'ै', 'ॉ', 'ो', 'ौ',

                      # Halant + signs
                      '्', 'ं', 'ः', 'ँ', 'ऽ',

                      # Marathi digits
                      '०', '१', '२', '३', '४', '५', '६', '७', '८', '९',

                      PADDING_TOKEN, END_TOKEN]


english_vocabulary = [START_TOKEN, ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', 
                        '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
                        ':', '<', '=', '>', '?', '@',
                        '[', '\\', ']', '^', '_', '`', 
                        'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l',
                        'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 
                        'y', 'z', 
                        '{', '|', '}', '~', PADDING_TOKEN, END_TOKEN]

In [3]:
index_to_marathi = {k:v for k,v in enumerate(marathi_vocabulary)}
marathi_to_index = {v:k for k,v in enumerate(marathi_vocabulary)}
index_to_english = {k:v for k,v in enumerate(english_vocabulary)}
english_to_index = {v:k for k,v in enumerate(english_vocabulary)}

In [4]:
with open(english_file, 'r', encoding='utf-8') as file:
    english_sentences = file.readlines()
with open(marathi_file, 'r', encoding='utf-8', errors='ignore') as file:
    marathi_sentences = file.readlines()

# Limit Number of sentences
TOTAL_SENTENCES = 200000
english_sentences = english_sentences[:TOTAL_SENTENCES]
marathi_sentences = marathi_sentences[:TOTAL_SENTENCES]
english_sentences = [sentence.rstrip('\n').lower() for sentence in english_sentences]
marathi_sentences = [sentence.rstrip('\n') for sentence in marathi_sentences]

In [5]:
english_sentences[:10]

['next few months are really crucial for us.',
 "sharad pawar, supriya sule, ajit pawar, praful patel, chhagan bhujbal and nawab malik from the ncp and ahmed patel, mallikarjun kharge, balasaheb thorat, prithviraj chavan from the congress were present in the meeting at the ncp chief's residence in delhi.",
 '"kanwal singh chauhan, the president of sonepat progressive farmers club, said, ""farmers are being misled."',
 'his demise is a huge loss for india.',
 'photographer and raconteur.',
 'he has won several competitions at the state and national level.',
 'this is a worrying increase.',
 'most of the victims were dalits.',
 'this has shocked the congress.',
 'this was not the idea.']

In [6]:
marathi_sentences[:10]

['पुढील तीन महिने आमच्यासाठी खूप महत्त्वपूर्ण आहेत.',
 'या बैठकील पक्षाध्यक्ष शरद पवार, केंद्रीय मंत्री प्रफुल्ल पटेल, उपमुख्यमंत्री अजित पवार, छगन भुजबळ, मधुकरराव पिचड, आर. आर. पाटील, सुनिल तटकरे, भास्कर जाधव आणि जितेंद्र आव्हाड हे नेते उपस्थित होते.',
 'वृत्तसंस्थेने दिलेल्या माहितीनुसार, विरोध करण्याऱ्या शेतकऱ्यांची दिशाभूल केली जात असल्याचा आरोप प्रगतीशील शेतकरी संघटना, सेनीपतचे अध्यक्ष कंवलसिंग चौहान यांनी केला.',
 'त्यांच्या मृत्यूमुळे भारताचे खूप मोठे नुकसान झाले.',
 'छायाचित्रकार आणि छायाचित्रकार.',
 'राज्य व राष्ट्रीय पातळीवरील अनेक मल्ल त्यांनी मेहनतीने घडविले.',
 'ही वाढ चिंताजनक तर आहेच.',
 'त्यातील बहुतांशी घटनांत पीडित व्यक्ती या दलित समाजाच्या होत्या.',
 'त्यामुळे काँग्रेसला मोठा धक्का बसला.',
 'हा कल्पनाविलास नव्हता.']

In [7]:
import numpy as np
PERCENTILE = 97
print( f"{PERCENTILE}th percentile length Marathi: {np.percentile([len(x) for x in marathi_sentences], PERCENTILE)}" )
print( f"{PERCENTILE}th percentile length English: {np.percentile([len(x) for x in english_sentences], PERCENTILE)}" )


97th percentile length Marathi: 183.0
97th percentile length English: 193.0


In [8]:
max_sequence_length = 200

def is_valid_tokens(sentence, vocab):
    for token in list(set(sentence)):
        if token not in vocab:
            return False
    return True

def is_valid_length(sentence, max_sequence_length):
    return len(list(sentence)) < (max_sequence_length - 1) # need to re-add the end token so leaving 1 space

valid_sentence_indicies = []
for index in range(len(marathi_sentences)):
    marathi_sentence, english_sentence = marathi_sentences[index], english_sentences[index]
    if is_valid_length(marathi_sentence, max_sequence_length) \
      and is_valid_length(english_sentence, max_sequence_length) \
      and is_valid_tokens(marathi_sentence, marathi_vocabulary):
        valid_sentence_indicies.append(index)

print(f"Number of sentences: {len(marathi_sentences)}")
print(f"Number of valid sentences: {len(valid_sentence_indicies)}")

Number of sentences: 200000
Number of valid sentences: 171771


In [9]:
marathi_sentences = [marathi_sentences[i] for i in valid_sentence_indicies]
english_sentences = [english_sentences[i] for i in valid_sentence_indicies]

In [10]:
marathi_sentences[:3]

['पुढील तीन महिने आमच्यासाठी खूप महत्त्वपूर्ण आहेत.',
 'वृत्तसंस्थेने दिलेल्या माहितीनुसार, विरोध करण्याऱ्या शेतकऱ्यांची दिशाभूल केली जात असल्याचा आरोप प्रगतीशील शेतकरी संघटना, सेनीपतचे अध्यक्ष कंवलसिंग चौहान यांनी केला.',
 'त्यांच्या मृत्यूमुळे भारताचे खूप मोठे नुकसान झाले.']

In [12]:
import torch

d_model = 512
batch_size = 30
ffn_hidden = 2048
num_heads = 8
drop_prob = 0.1
num_layers = 1
max_sequence_length = 200
mr_vocab_size = len(marathi_vocabulary)

transformer = Transformer(d_model, 
                          ffn_hidden,
                          num_heads, 
                          drop_prob, 
                          num_layers, 
                          max_sequence_length,
                          mr_vocab_size,
                          english_to_index,
                          marathi_to_index,
                          START_TOKEN, 
                          END_TOKEN, 
                          PADDING_TOKEN)

In [13]:
transformer

Transformer(
  (encoder): Encoder(
    (sentence_embedding): SentenceEmbedding(
      (embedding): Embedding(71, 512)
      (position_encoder): PositionalEncoding()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (layers): SequentialEncoder(
      (0): EncoderLayer(
        (attention): MultiHeadAttention(
          (qkv_layer): Linear(in_features=512, out_features=1536, bias=True)
          (linear_layer): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm1): LayerNormalization()
        (dropout1): Dropout(p=0.1, inplace=False)
        (ffn): PositionwiseFeedForward(
          (linear1): Linear(in_features=512, out_features=2048, bias=True)
          (linear2): Linear(in_features=2048, out_features=512, bias=True)
          (relu): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (norm2): LayerNormalization()
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (decoder): Decoder(
    (sentence_embedding):

In [14]:
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):

    def __init__(self, english_sentences, marathi_sentences):
        self.english_sentences = english_sentences
        self.marathi_sentences = marathi_sentences

    def __len__(self):
        return len(self.english_sentences)

    def __getitem__(self, idx):
        return self.english_sentences[idx], self.marathi_sentences[idx]

In [15]:
dataset = TextDataset(english_sentences, marathi_sentences)

In [16]:
len(dataset)

171771

In [17]:
dataset[1]

('"kanwal singh chauhan, the president of sonepat progressive farmers club, said, ""farmers are being misled."',
 'वृत्तसंस्थेने दिलेल्या माहितीनुसार, विरोध करण्याऱ्या शेतकऱ्यांची दिशाभूल केली जात असल्याचा आरोप प्रगतीशील शेतकरी संघटना, सेनीपतचे अध्यक्ष कंवलसिंग चौहान यांनी केला.')

In [18]:
train_loader = DataLoader(dataset, batch_size)
iterator = iter(train_loader)

In [21]:
for batch_num, batch in enumerate(iterator):
    print(batch)
    if batch_num > 3:
        break

[('meteorologists say that there will be a possibility of heavy rains in some areas too.', 'many vehicles were stranded in the water-logged roads.', 'ayodhya verdict: what is the case about?', 'reviewed the security situation in delhi.', 'this was indeed a historic moment in the history of indian cricket.', 'without having a majority, how can the bjp do so?', 'bjp leaders were also present.', 'this hasnt been an overnight phenomenon.', 'as a result, the traffic on the route was completely disrupted.', 'gurbaz was the top-scorer for afghanistan with 79 runs off 52 balls.', 'you dont look too good.', 'so i have no issue.', 'i went in and got through.', "ranveer singh and deepika padukone's marriage preparations are already on in full swing.", 'if you dont succeed, again try.', 'we then developed it.', 'all indians were required to carry identification and registration cards at all times.', 'the traffic on the highway was disrupted for sometime due to the protests.', 'but we cant do nothi

In [22]:
from torch import nn

criterian = nn.CrossEntropyLoss(ignore_index=marathi_to_index[PADDING_TOKEN],
                                reduction='none')

# When computing the loss, we are ignoring cases when the label is the padding token
for params in transformer.parameters():
    if params.dim() > 1:
        nn.init.xavier_uniform_(params)

optim = torch.optim.Adam(transformer.parameters(), lr=1e-4)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [23]:
NEG_INFTY = -1e9

def create_masks(eng_batch, mr_batch):
    num_sentences = len(eng_batch)
    look_ahead_mask = torch.full([max_sequence_length, max_sequence_length] , True)
    look_ahead_mask = torch.triu(look_ahead_mask, diagonal=1)
    encoder_padding_mask = torch.full([num_sentences, max_sequence_length, max_sequence_length] , False)
    decoder_padding_mask_self_attention = torch.full([num_sentences, max_sequence_length, max_sequence_length] , False)
    decoder_padding_mask_cross_attention = torch.full([num_sentences, max_sequence_length, max_sequence_length] , False)

    for idx in range(num_sentences):
      eng_sentence_length, mr_sentence_length = len(eng_batch[idx]), len(mr_batch[idx])
      eng_chars_to_padding_mask = np.arange(eng_sentence_length + 1, max_sequence_length)
      mr_chars_to_padding_mask = np.arange(mr_sentence_length + 1, max_sequence_length)
      encoder_padding_mask[idx, :, eng_chars_to_padding_mask] = True
      encoder_padding_mask[idx, eng_chars_to_padding_mask, :] = True
      decoder_padding_mask_self_attention[idx, :, mr_chars_to_padding_mask] = True
      decoder_padding_mask_self_attention[idx, mr_chars_to_padding_mask, :] = True
      decoder_padding_mask_cross_attention[idx, :, eng_chars_to_padding_mask] = True
      decoder_padding_mask_cross_attention[idx, mr_chars_to_padding_mask, :] = True

    encoder_self_attention_mask = torch.where(encoder_padding_mask, NEG_INFTY, 0)
    decoder_self_attention_mask =  torch.where(look_ahead_mask + decoder_padding_mask_self_attention, NEG_INFTY, 0)
    decoder_cross_attention_mask = torch.where(decoder_padding_mask_cross_attention, NEG_INFTY, 0)
    return encoder_self_attention_mask, decoder_self_attention_mask, decoder_cross_attention_mask

Modify mask such that the padding tokens cannot look ahead.
In Encoder, tokens before it should be -1e9 while tokens after it should be -inf.
 

Note the target mask starts with 2 rows of non masked items: https://github.com/SamLynnEvans/Transformer/blob/master/Beam.py#L55


In [25]:
transformer.train()
transformer.to(device)
total_loss = 0
num_epochs = 10

for epoch in range(num_epochs):
    print(f"Epoch {epoch}")
    iterator = iter(train_loader)
    for batch_num, batch in enumerate(iterator):
        transformer.train()
        eng_batch, mr_batch = batch
        encoder_self_attention_mask, decoder_self_attention_mask, decoder_cross_attention_mask = create_masks(eng_batch, mr_batch)
        optim.zero_grad()
        mr_predictions = transformer(eng_batch,
                                     mr_batch,
                                     encoder_self_attention_mask.to(device), 
                                     decoder_self_attention_mask.to(device), 
                                     decoder_cross_attention_mask.to(device),
                                     enc_start_token=False,
                                     enc_end_token=False,
                                     dec_start_token=True,
                                     dec_end_token=True)
        labels = transformer.decoder.sentence_embedding.batch_tokenize(mr_batch, start_token=False, end_token=True)
        loss = criterian(
            mr_predictions.view(-1, mr_vocab_size).to(device),
            labels.view(-1).to(device)
        ).to(device)
        valid_indicies = torch.where(labels.view(-1) == marathi_to_index[PADDING_TOKEN], False, True)
        loss = loss.sum() / valid_indicies.sum()
        loss.backward()
        optim.step()
        #train_losses.append(loss.item())
        if batch_num % 100 == 0:
            print(f"Iteration {batch_num} : {loss.item()}")
            print(f"English: {eng_batch[0]}")
            print(f"Marathi Translation: {mr_batch[0]}")
            mr_sentence_predicted = torch.argmax(mr_predictions[0], axis=1)
            predicted_sentence = ""
            for idx in mr_sentence_predicted:
              if idx == marathi_to_index[END_TOKEN]:
                break
              predicted_sentence += index_to_marathi[idx.item()]
            print(f"Marathi Prediction: {predicted_sentence}")


            transformer.eval()
            mr_sentence = ("",)
            eng_sentence = ("should we go to the mall?",)
            for word_counter in range(max_sequence_length):
                encoder_self_attention_mask, decoder_self_attention_mask, decoder_cross_attention_mask= create_masks(eng_sentence, mr_sentence)
                predictions = transformer(eng_sentence,
                                          mr_sentence,
                                          encoder_self_attention_mask.to(device), 
                                          decoder_self_attention_mask.to(device), 
                                          decoder_cross_attention_mask.to(device),
                                          enc_start_token=False,
                                          enc_end_token=False,
                                          dec_start_token=True,
                                          dec_end_token=False)
                next_token_prob_distribution = predictions[0][word_counter] # not actual probs
                next_token_index = torch.argmax(next_token_prob_distribution).item()
                next_token = index_to_marathi[next_token_index]
                mr_sentence = (mr_sentence[0] + next_token, )
                if next_token == END_TOKEN:
                  break
            
            print(f"Evaluation translation (should we go to the mall?) : {mr_sentence}")
            print("-------------------------------------------")

Epoch 0
Iteration 0 : 1.9132499694824219
English: next few months are really crucial for us.
Marathi Translation: पुढील तीन महिने आमच्यासाठी खूप महत्त्वपूर्ण आहेत.
Marathi Prediction: त्ढेल स्न साालालसणचीया  ठी साप कात्त वा र ण कहे..
Evaluation translation (should we go to the mall?) : ('तुम्ही काय काय काय काय काय का?<END>',)
-------------------------------------------
Iteration 100 : 2.024513006210327
English: citation of awardees of national awards for the empowerment of persons with disabilities (divyangjan) 2019
Marathi Translation: यावेळी विविध 14 श्रेणींमध्ये दिव्यांगासाठीच्या राष्ट्रीय पुरस्कारांचे वितरण करण्यात आले.
Marathi Prediction: ता ेळी सिधिधाम  मेरेसा चध्ये मेवसहा नाताठी ्या पाज्ट्रप  सारा्तार ंच् मिध् ासे््यात आले 
Evaluation translation (should we go to the mall?) : ('त्यांना काय काय काय काय काय का?<END>',)
-------------------------------------------
Iteration 200 : 1.9560987949371338
English: vegetable market
Marathi Translation: किरकोळ बाजारात भाजी
Marathi Prediction

## Inference

In [26]:
transformer.eval()
def translate(eng_sentence):
  eng_sentence = (eng_sentence,)
  mr_sentence = ("",)
  for word_counter in range(max_sequence_length):
    encoder_self_attention_mask, decoder_self_attention_mask, decoder_cross_attention_mask= create_masks(eng_sentence, mr_sentence)
    predictions = transformer(eng_sentence,
                              mr_sentence,
                              encoder_self_attention_mask.to(device), 
                              decoder_self_attention_mask.to(device), 
                              decoder_cross_attention_mask.to(device),
                              enc_start_token=False,
                              enc_end_token=False,
                              dec_start_token=True,
                              dec_end_token=False)
    next_token_prob_distribution = predictions[0][word_counter]
    next_token_index = torch.argmax(next_token_prob_distribution).item()
    next_token = index_to_marathi[next_token_index]
    mr_sentence = (mr_sentence[0] + next_token, )
    if next_token == END_TOKEN:
      break
  return mr_sentence[0]

In [ ]:
translation = translate("what should we do when the day starts?")
print(translation)

त्यांना काय करण्यासाठी काय करण्यासाठी काय करावे?<END>


In [ ]:
translation = translate("how is this the truth?")
print(translation)

त्यांच्या काय करायची कसे कसे आहे?<END>


In [ ]:
translation = translate("the world is a large place with different people")
print(translation)

पालकांनी पालकांनी पालकांनी सांगितले पाहिजे.<END>


In [ ]:
translation = translate("my name is ajay")
print(translation)

माझ्या मुंबई माझ्या माझा आहे.<END>


In [ ]:
translation = translate("i cannot stand this smell")
print(translation)

मी कायदा माझ्या प्रकारची नाही<END>


In [32]:
translation = translate("noodles are the best")
print(translation)

कोणत्याही कोणतीही काळजी<END>


In [33]:
translation = translate("why care about this?")
print(translation)

त्यांच्या का का का का ?<END>


In [ ]:
translation = translate("this is the best thing ever")
print(translation)

हे सर्वात मोठा आहे<END>


In [ ]:
translation = translate("i am here")
print(translation)

मी त्यांच्या माझ्या आहे.<END>


In [ ]:
translation = translate("click this")
print(translation)

कायद्याची प्रकार<END>


In [37]:
translation = translate("where is the mall?")
print(translation)

काय आहे काय आहे?<END>


In [38]:
translation = translate("what should we do?")
print(translation)

काय आहे काय करायचे?<END>


In [39]:
translation = translate("today, what should we do")
print(translation)

आज आपल्या काय करायचे प्रयत्न करावे लागले आहे<END>


In [ ]:
translation = translate("why did they activate?")
print(translation)

त्यांना का का का का ?<END>


In [ ]:
translation = translate("why did they do this?")
print(translation)

त्यांना का का का का का का ?<END>


In [ ]:
translation = translate("i am well.")
print(translation)

मी त्याची मी आहे.<END>


In [43]:
translation = translate("whats the word on the street?")
print(translation)

काय आहे काय करायचे काय आहे?<END>
